<a href="https://colab.research.google.com/github/pnperl/Travel_Router/blob/main/Travel_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ============================================================
# SMART INDIAN TRAVEL ENGINE - PHASE 4
# ============================================================

!pip -q install pandas networkx tabulate ipywidgets plotly

import pandas as pd
import networkx as nx
import plotly.express as px

from tabulate import tabulate
from datetime import datetime, timedelta

import ipywidgets as widgets

from IPython.display import display, HTML, clear_output

# ============================================================
# CONFIG
# ============================================================

MAX_HOPS = 4

MAX_TOTAL_HOURS = 36

MAX_LAYOVER_HOURS = 8

# ============================================================
# CITY → STATION MAP
# ============================================================

CITY_TO_STATIONS = {

    "JAMNAGAR": {
        "rail": ["JAM", "HAPA"]
    },

    "RAJKOT": {
        "rail": ["RJT"]
    },

    "AHMEDABAD": {
        "rail": ["ADI", "SBIB"]
    },

    "SURAT": {
        "rail": ["ST"]
    },

    "VADODARA": {
        "rail": ["BRC"]
    },

    "MUMBAI": {

        "rail": [
            "MMCT",
            "BDTS",
            "DDR",
            "LTT",
            "CSMT",
            "PNVL"
        ]
    }
}

# ============================================================
# MUMBAI TERMINAL PREFERENCE
# ============================================================

TERMINAL_BONUS = {

    "MMCT": 30,
    "BDTS": 20,
    "DDR": 15,
    "LTT": 5,
    "PNVL": -10
}

# ============================================================
# MAJOR JUNCTION RISK
# ============================================================

MAJOR_JUNCTIONS = {

    "ADI": 20,
    "MMCT": 25,
    "BDTS": 15,
    "PNVL": 10
}

# ============================================================
# SAMPLE MULTI-MODAL DATA
# ============================================================

TRANSPORTS = [

    # ========================================================
    # BUS
    # ========================================================

    {
        "mode": "BUS",
        "name": "Volvo AC Sleeper",
        "from": "JAM",
        "to": "ADI",
        "dep": "14:00",
        "arr": "20:00",
        "class": "AC_SLEEPER",
        "availability": 24
    },

    {
        "mode": "BUS",
        "name": "Rajkot Shuttle",
        "from": "JAM",
        "to": "RJT",
        "dep": "16:00",
        "arr": "18:30",
        "class": "AC_SEATER",
        "availability": 40
    },

    # ========================================================
    # TRAINS
    # ========================================================

    {
        "mode": "TRAIN",
        "name": "Garib Rath",
        "from": "ADI",
        "to": "BDTS",
        "dep": "23:00",
        "arr": "08:00",
        "class": "3A",
        "availability": 36
    },

    {
        "mode": "TRAIN",
        "name": "Gujarat Mail",
        "from": "ADI",
        "to": "MMCT",
        "dep": "22:00",
        "arr": "06:00",
        "class": "3A",
        "availability": 28
    },

    {
        "mode": "TRAIN",
        "name": "Lok Shakti",
        "from": "ADI",
        "to": "BDTS",
        "dep": "20:30",
        "arr": "07:00",
        "class": "3A",
        "availability": 14
    },

    {
        "mode": "TRAIN",
        "name": "Humsafar Express",
        "from": "JAM",
        "to": "BDTS",
        "dep": "19:30",
        "arr": "09:30",
        "class": "3A",
        "availability": 12
    },

    {
        "mode": "TRAIN",
        "name": "Saurashtra Janta",
        "from": "JAM",
        "to": "BDTS",
        "dep": "21:45",
        "arr": "12:00",
        "class": "SL",
        "availability": 72
    },

    {
        "mode": "TRAIN",
        "name": "Saurashtra Mail",
        "from": "RJT",
        "to": "DDR",
        "dep": "21:00",
        "arr": "10:30",
        "class": "3A",
        "availability": 9
    }
]

df = pd.DataFrame(TRANSPORTS)

# ============================================================
# UI
# ============================================================

cities = sorted(CITY_TO_STATIONS.keys())

source_widget = widgets.Combobox(
    options=cities,
    description='From:',
    ensure_option=True
)

dest_widget = widgets.Combobox(
    options=cities,
    description='To:',
    ensure_option=True
)

group_widget = widgets.IntSlider(
    value=10,
    min=1,
    max=30,
    description='Passengers:'
)

today = datetime.today()

start_widget = widgets.DatePicker(
    description='Start',
    value=today
)

end_widget = widgets.DatePicker(
    description='End',
    value=today + timedelta(days=7)
)

display(source_widget)
display(dest_widget)
display(group_widget)
display(start_widget)
display(end_widget)

# ============================================================
# HELPERS
# ============================================================

def parse_time(t):

    return datetime.strptime(t, "%H:%M")

def layover_minutes(arr, dep):

    arr_dt = parse_time(arr)
    dep_dt = parse_time(dep)

    if dep_dt < arr_dt:
        dep_dt += timedelta(days=1)

    return int((dep_dt - arr_dt).seconds / 60)

def daterange(start_date, end_date):

    for n in range(
        int((end_date - start_date).days) + 1
    ):
        yield start_date + timedelta(n)

def total_journey_hours(edges):

    start = parse_time(edges[0]["dep"])

    end = parse_time(edges[-1]["arr"])

    if end < start:
        end += timedelta(days=1)

    return round(
        (end - start).seconds / 3600,
        1
    )

# ============================================================
# CONFIRMATION PROBABILITY
# ============================================================

def confirmation_probability(avail):

    if avail >= 50:
        return "99%"

    if avail >= 20:
        return "95%"

    if avail >= 10:
        return "85%"

    if avail >= 5:
        return "65%"

    return "LOW"

# ============================================================
# SPLIT OPTIMIZER
# ============================================================

def split_optimizer(required, available):

    if available >= required:

        return (
            f"{required} together confirmed"
        )

    remaining = required - available

    if remaining <= 2:

        return (
            f"{available} confirmed + "
            f"{remaining} Tatkal"
        )

    if remaining <= 4:

        return (
            f"{available} confirmed + "
            f"{remaining} nearby train"
        )

    return (
        f"Split group recommended "
        f"({available} + {remaining})"
    )

# ============================================================
# COMFORT LABEL
# ============================================================

def comfort_label(score):

    if score >= 250:
        return "Excellent"

    if score >= 180:
        return "Very Good"

    if score >= 120:
        return "Good"

    if score >= 80:
        return "Average"

    return "Poor"

# ============================================================
# GRAPH ENGINE
# ============================================================

def build_graph():

    G = nx.DiGraph()

    for _, row in df.iterrows():

        G.add_edge(

            row["from"],
            row["to"],

            details=dict(row)
        )

    return G

# ============================================================
# UNIVERSAL ROUTE SEARCH
# ============================================================

def find_all_routes(

    G,
    source_stations,
    dest_stations

):

    all_paths = []

    for src in source_stations:

        for dst in dest_stations:

            if src not in G.nodes:
                continue

            if dst not in G.nodes:
                continue

            try:

                paths = list(

                    nx.all_simple_paths(

                        G,

                        source=src,

                        target=dst,

                        cutoff=MAX_HOPS
                    )
                )

                all_paths.extend(paths)

            except:

                continue

    return all_paths

# ============================================================
# PATH TO EDGE CONVERTER
# ============================================================

def path_to_edges(G, path):

    edges = []

    for i in range(len(path) - 1):

        edge = G[
            path[i]
        ][
            path[i + 1]
        ]["details"]

        edges.append(edge)

    return edges

# ============================================================
# ROUTE VALIDATOR
# ============================================================

def is_reasonable_route(edges):

    total_layover = 0

    for i in range(len(edges) - 1):

        lay = layover_minutes(

            edges[i]["arr"],
            edges[i + 1]["dep"]
        )

        total_layover += lay

        if lay < 45:
            return False

        if lay > 480:
            return False

    journey = total_journey_hours(edges)

    if journey > MAX_TOTAL_HOURS:
        return False

    return True

# ============================================================
# BALANCED SCORING ENGINE
# ============================================================

def score_route(edges, passengers):

    score = 0

    min_avail = min(
        [x["availability"] for x in edges]
    )

    # ========================================================
    # GROUP AVAILABILITY
    # ========================================================

    if min_avail >= passengers:
        score += 120

    elif min_avail >= passengers * 0.7:
        score += 70

    elif min_avail >= passengers * 0.5:
        score += 40

    else:
        score -= 40

    score += min(min_avail, 50)

    # ========================================================
    # HOPS
    # ========================================================

    hops = len(edges) - 1

    if hops == 0:
        score += 60

    elif hops == 1:
        score += 25

    elif hops >= 3:
        score -= 50

    # ========================================================
    # AC BONUS
    # ========================================================

    for edge in edges:

        if edge["class"] in [
            "3A",
            "2A",
            "AC_SLEEPER"
        ]:
            score += 35

        elif edge["class"] == "SL":
            score += 10

    # ========================================================
    # NIGHT BONUS
    # ========================================================

    for edge in edges:

        dep_hr = int(edge["dep"].split(":")[0])

        if dep_hr >= 19:
            score += 20

    # ========================================================
    # LAYOVER BALANCE
    # ========================================================

    for i in range(len(edges) - 1):

        lay = layover_minutes(

            edges[i]["arr"],
            edges[i + 1]["dep"]
        )

        if 60 <= lay <= 180:
            score += 45

        elif 180 < lay <= 300:
            score += 15

        elif lay < 45:
            score -= 120

        elif lay > 420:
            score -= 70

        hub = edges[i]["to"]

        risk = MAJOR_JUNCTIONS.get(hub, 5)

        score -= risk

    # ========================================================
    # BUS PENALTY
    # ========================================================

    bus_count = len([
        x for x in edges
        if x["mode"] == "BUS"
    ])

    score -= bus_count * 10

    # ========================================================
    # DAYTIME SLEEPER PENALTY
    # ========================================================

    for edge in edges:

        dep_hr = int(edge["dep"].split(":")[0])

        if (
            edge["class"] == "SL"
            and
            10 <= dep_hr <= 17
        ):
            score -= 40

    # ========================================================
    # JOURNEY HOURS
    # ========================================================

    journey_hours = total_journey_hours(edges)

    if journey_hours <= 12:
        score += 40

    elif journey_hours <= 18:
        score += 10

    elif journey_hours > 24:
        score -= 60

    # ========================================================
    # MORNING ARRIVAL BONUS
    # ========================================================

    arr_hr = int(
        edges[-1]["arr"].split(":")[0]
    )

    if 5 <= arr_hr <= 9:
        score += 35

    # ========================================================
    # TERMINAL BONUS
    # ========================================================

    dest_bonus = TERMINAL_BONUS.get(
        edges[-1]["to"],
        0
    )

    score += dest_bonus

    return score

# ============================================================
# MAIN SEARCH ENGINE
# ============================================================

def find_routes():

    clear_output(wait=True)

    display(source_widget)
    display(dest_widget)
    display(group_widget)
    display(start_widget)
    display(end_widget)
    display(search_button)

    source_city = source_widget.value.upper()
    dest_city = dest_widget.value.upper()

    passengers = group_widget.value

    if source_city not in CITY_TO_STATIONS:
        print("Invalid source city")
        return

    if dest_city not in CITY_TO_STATIONS:
        print("Invalid destination city")
        return

    source_stations = (
        CITY_TO_STATIONS[source_city]["rail"]
    )

    dest_stations = (
        CITY_TO_STATIONS[dest_city]["rail"]
    )

    G = build_graph()

    all_paths = find_all_routes(

        G,
        source_stations,
        dest_stations
    )

    results = []

    total_days = (
        end_widget.value -
        start_widget.value
    ).days + 1

    progress = widgets.IntProgress(
        value=0,
        min=0,
        max=total_days,
        description='Searching:',
        bar_style='info'
    )

    display(progress)

    for idx, day in enumerate(

        daterange(
            start_widget.value,
            end_widget.value
        )

    ):

        progress.value = idx + 1

        for path in all_paths:

            edges = path_to_edges(G, path)

            if not is_reasonable_route(edges):
                continue

            score = score_route(
                edges,
                passengers
            )

            min_avail = min(
                [
                    x["availability"]
                    for x in edges
                ]
            )

            timeline = ""

            for edge in edges:

                timeline += (
                    f"{edge['from']} "
                    f"→ "
                    f"{edge['to']} "
                    f"({edge['mode']})<br>"
                )

            results.append({

                "date":
                    str(day),

                "route":
                    " → ".join(path),

                "availability":
                    min_avail,

                "confirmation":
                    confirmation_probability(
                        min_avail
                    ),

                "comfort":
                    comfort_label(score),

                "split_plan":
                    split_optimizer(
                        passengers,
                        min_avail
                    ),

                "score":
                    score,

                "timeline":
                    timeline
            })

    if len(results) == 0:

        print("No routes found")
        return

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(

        by="score",
        ascending=False
    )

    # ========================================================
    # BEST DATE
    # ========================================================

    best_date = (
        results_df.groupby("date")["score"]
        .mean()
        .sort_values(ascending=False)
        .index[0]
    )

    print("\n")
    print("=" * 100)
    print("BEST DATE TO TRAVEL")
    print("=" * 100)

    print(best_date)

    # ========================================================
    # TABLE
    # ========================================================

    print("\n")
    print("=" * 120)
    print("TOP ROUTES")
    print("=" * 120)

    print(tabulate(

        results_df[
            [
                "date",
                "route",
                "availability",
                "confirmation",
                "comfort",
                "score",
                "split_plan"
            ]
        ].head(10),

        headers="keys",
        tablefmt="fancy_grid",
        showindex=False
    ))

    # ========================================================
    # CHART
    # ========================================================

    fig = px.bar(

        results_df.head(10),

        x="route",
        y="score",

        hover_data=[
            "availability",
            "confirmation",
            "comfort"
        ],

        title="Best Comfortable Routes"
    )

    fig.show()

# ============================================================
# SEARCH BUTTON
# ============================================================

search_button = widgets.Button(

    description='Search Comfortable Routes',

    button_style='success',

    icon='search',

    layout=widgets.Layout(
        width='300px',
        height='50px'
    )
)

search_button.on_click(
    lambda x: find_routes()
)

display(search_button)

Combobox(value='JAMNAGAR', description='From:', ensure_option=True, options=('AHMEDABAD', 'JAMNAGAR', 'MUMBAI'…

Combobox(value='MUMBAI', description='To:', ensure_option=True, options=('AHMEDABAD', 'JAMNAGAR', 'MUMBAI', 'R…

IntSlider(value=10, description='Passengers:', max=30, min=1)

DatePicker(value=datetime.datetime(2026, 5, 12, 16, 6, 28, 429220), description='Start')

DatePicker(value=datetime.datetime(2026, 5, 19, 16, 6, 28, 429220), description='End')

Button(button_style='success', description='Search Comfortable Routes', icon='search', layout=Layout(height='5…

IntProgress(value=0, bar_style='info', description='Searching:', max=8)



BEST DATE TO TRAVEL
2026-05-12 16:06:28.429220


TOP ROUTES
╒════════════════════════════╤══════════════════╤════════════════╤════════════════╤═══════════╤═════════╤═══════════════════════╕
│ date                       │ route            │   availability │ confirmation   │ comfort   │   score │ split_plan            │
╞════════════════════════════╪══════════════════╪════════════════╪════════════════╪═══════════╪═════════╪═══════════════════════╡
│ 2026-05-12 16:06:28.429220 │ JAM → ADI → MMCT │             24 │ 95%            │ Excellent │     349 │ 10 together confirmed │
├────────────────────────────┼──────────────────┼────────────────┼────────────────┼───────────┼─────────┼───────────────────────┤
│ 2026-05-13 16:06:28.429220 │ JAM → ADI → MMCT │             24 │ 95%            │ Excellent │     349 │ 10 together confirmed │
├────────────────────────────┼──────────────────┼────────────────┼────────────────┼───────────┼─────────┼───────────────────────┤
│ 2026-05-14 16:06:28.429220